In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

landing_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "landing/orders/"
)

bronze_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/orders/"
)

checkpoint_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/bronze_orders/"
)

schema_orders_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "checkpoints/schemas/bronze_orders/"
)

# ============================================================
# 2. SCHÉMA DES DONNÉES SOURCE
# ============================================================

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_time", TimestampType(), True),
    StructField("destination_city", StringType(), True),
    StructField("destination_country", StringType(), True),
    StructField("total_amount", DecimalType(10, 2), True),
    StructField("currency", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])

# ============================================================
# 3. LECTURE INCRÉMENTALE AVEC AUTO LOADER
# ============================================================

orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_orders_path)
    .option("header", "true")
    .schema(orders_schema)
    .load(landing_orders_path)
)

# ============================================================
# 4. AJOUT DES MÉTADONNÉES BRONZE
# ============================================================

bronze_orders_df = (
    orders_stream
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "batch_id",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"(batch_\d+)",
            1
        )
    )
)

# ============================================================
# 5. ÉCRITURE DELTA BRONZE
# ============================================================

query = (
    bronze_orders_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_orders_path
    )
    .option(
        "path",
        bronze_orders_path
    )
    .trigger(availableNow=True)
    .start()
)

# Attendre la fin du batch disponible
query.awaitTermination()

print("Ingestion Bronze orders terminée.")

# ============================================================
# 6. LECTURE DE LA TABLE DELTA BRONZE
# ============================================================

bronze_orders = (
    spark.read
    .format("delta")
    .load(bronze_orders_path)
)

print(
    "Nombre de lignes Bronze :",
    bronze_orders.count()
)

display(
    bronze_orders
    .orderBy("order_id")
)

# ============================================================
# 7. VÉRIFICATION DES MÉTADONNÉES
# ============================================================

display(
    bronze_orders.select(
        "order_id",
        "source_file",
        "batch_id",
        "ingestion_timestamp"
    )
    .orderBy("order_id")
)